In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import spacy
from spacy import displacy
import networkx as nx
import os
import matplotlib.pyplot as plt
import scipy
import re
import country_converter as coco

In [2]:
# Load spacy English module

NER = spacy.load("en_core_web_sm")

In [3]:
# Load 20C text file
with open("20C_events_article_Wiki.txt", 'r', errors='ignore') as file:
    lines = file.readlines()

In [4]:
# Cutting out references and further reading sections
try:
    cut_index = lines.index("See also[edit]\n")  # everything from this on
    trimmed_data = ''.join(lines[:cut_index])
except ValueError:
    trimmed_data = ''.join(lines)  

In [5]:
print(trimmed_data[-500:])

e possibility of human extinction, although not overnight but over several decades.[273] This prompted many nations to negotiate and sign the Kyoto treaty, which set mandatory limits on carbon dioxide emissions.[274]
The celebration of the 20th centuryâ€™s ending expressed the popular opinion that New Year's Eve 1999 and New Year's Day 2000 marked the turn of the millennium, while strictly speaking the 20th century ended on New Year's Eve 2000 and the 21st century began on New Year's Day 2001.




this worked, this is where the main article ends

Some of the main countries are referred to as different names in the text than the CSV, and especially the main countries. Russia:Soviet Union/USSR, United States: America/US, United Kingdom: Britain/Great Britain.
I've looked through all cases of 'America' and none refer to latin or central america.

In [6]:
# Loading countries csv
countries = pd.read_csv('countries.csv', index_col=False)
official_countries = countries['Country'].dropna().unique().tolist()

In [7]:
# getting country demonyms based off data from https://gist.github.com/consti/e2c7ddc64f0aa044a8b4fcd28dba0700 
df_demons = pd.read_csv('emoji_country_nationality_list.csv', usecols=['Name', 'Demonym 1', 'Demonym 2', 'Demonym 3'])
df_demons = df_demons.fillna('')
df_demons.head()

,Name,Demonym 1,Demonym 2,Demonym 3
0,Andorra,Andorran,,
1,United Arab Emirates,Emirian,Emirati,
2,Afghanistan,Afghani,Afghan,
3,Antigua and Barbuda,Antiguan,,
4,Anguilla,Anguillan,,


In [8]:
# Demonym to country mapping
demonym_dict = {}   # empty list for dict

# loop through rows of demonym df, get country name from name column and convert to lower
for _, row in df_demons.iterrows(): 
    country = row['Name'].lower()
    for col in ['Demonym 1', 'Demonym 2', 'Demonym 3']:     # do it over all columns
        dem = row[col].strip().lower()
        if dem:
            demonym_dict[dem] = country

In [9]:
# integrate with country converter for other aliases 
cc = coco.CountryConverter()

# get the iso3 code, then convert it to official country name in lower case
def get_official(country_name):
    iso3 = cc.convert(names=country_name, to='ISO3')
    if isinstance(iso3, list):
        iso3 = iso3[0] if iso3 else None
    if not iso3 or iso3 == 'not found':
        return None
    name_short = cc.convert(names=iso3, to='name_short')
    if isinstance(name_short, list):
        name_short = name_short[0] if name_short else None
    return name_short.lower() if isinstance(name_short, str) else None

In [10]:
# Convert base demonym mapping to official standard names:
expanded_aliases = {}

# loop through key-value pairs in demonym dictionary
for dem, country in demonym_dict.items():
    std = get_official(country)     # get standard official name
    if std:
        expanded_aliases[dem] = std     # save to expanded aliases dict

congo (republic of) not found in regex


In [11]:
# Add official country names and their lowercase forms
for name in official_countries:
    std_name = get_official(name)
    if std_name:
        expanded_aliases[std_name] = std_name       # key=standard name itself
        expanded_aliases[name.lower()] = std_name 

Abkhazia not found in regex
Artsakh not found in regex
Akrotiri not found in regex
Ascension Island not found in regex
Ashmore and Cartier Islands not found in regex
Bajo Nuveo Bank not found in regex
Baker Island not found in regex
Canary Islands not found in regex
Ceuta not found in regex
Chatham Islands not found in regex
Clipperton Island not found in regex
Coral Sea Islands not found in regex
Corsica not found in regex
Crimea not found in regex
Dagestan not found in regex
Dhekelia not found in regex
Donetsk not found in regex
Ducie Island not found in regex
Easter Island not found in regex
Galápagos Islands not found in regex
Gilgit-Baltistan not found in regex
Gagauzia not found in regex
Guantanamo Bank not found in regex
Henderson Island not found in regex
Herm not found in regex
Howland Island not found in regex
Jan Mayen not found in regex
Jarvis Island not found in regex
Jeju Island not found in regex
Jervis Bay not found in regex
Jethou not found in regex
Johnston Atoll not 

In [12]:
print(expanded_aliases)

{'andorran': 'andorra', 'emirian': 'united arab emirates', 'emirati': 'united arab emirates', 'afghani': 'afghanistan', 'afghan': 'afghanistan', 'antiguan': 'antigua and barbuda', 'anguillan': 'anguilla', 'albanian': 'albania', 'alabanian': 'albania', 'armenian': 'armenia', 'hayastani': 'armenia', 'angolan': 'angola', 'antarctic': 'antarctica', 'argentine': 'argentina', 'argentinian': 'argentina', 'argentinean': 'argentina', 'samoan': 'samoa', 'austrian': 'austria', 'australian': 'australia', 'arubian': 'aruba', 'ålandic': 'aland islands', 'ålandish': 'aland islands', 'azerbaijani': 'azerbaijan', 'bosnian': 'bosnia and herzegovina', 'herzegovinian': 'bosnia and herzegovina', 'barbadian': 'barbados', 'barbadan': 'barbados', 'bajan': 'barbados', 'bangladeshi': 'bangladesh', 'belgian': 'belgium', 'burkinabe': 'burkina faso', 'bulgarian': 'bulgaria', 'bahrainian': 'bahrain', 'burundian': 'burundi', 'beninese': 'benin', 'barthélemois': 'st. barths', 'bermudan': 'bermuda', 'bruneian': 'brune

In [13]:
# Adding manual aliases
manual_aliases = {
    'britain': 'united kingdom',
    'great britain': 'united kingdom',
    'british': 'united kingdom',
    'america': 'united states',
    'u.s.': 'united states',
    'u.s.a.': 'united states',
    'south korea': 'korea, republic of',
    'north korea': 'korea, democratic people\'s republic of',
    'ussr': 'russia',
    'soviet': 'russia'
}

# Update expanded_aliases with manual entries (lowercase keys)
for alias, std_name in manual_aliases.items():
    expanded_aliases[alias.lower()] = std_name.lower()

In [14]:
# Process book with spacy NER
doc = NER(trimmed_data)

Getting named entity list per sentence

In [15]:
df_sentences = [] # empty shell to store results

# Loop through sentences, get entity list for each sentence
for sent in doc.sents:
    entity_list = [ent.text for ent in sent.ents]
    df_sentences.append({"sentence": sent, "entities": entity_list})
    
df_sentences = pd.DataFrame(df_sentences)

In [16]:
df_sentences.head(10)

,sentence,entities
0,"(\n\n\n\n, Key, events, of, the, 20th, century...","[the 20th century - Wikipedia, Jump, Main, Mai..."
1,"(1.4.1, \n, The, war, in, Europe, \n\n\n\n\n\n...","[1.4.1, Europe, 1.4.2, Blitzkrieg, 1.4.3, Oper..."
2,"(The, World, Wars, sparked, tension, between, ...","[the Cold War, the Space Race, the World Wide ..."
3,"(These, advancements, have, played, a, signifi...","[the 21st century, today]"
4,"(Historic, events, in, the, 20th, century[edit...","[Historic, 20th, World, Main, the 20th century]"
5,"(The, 1900s, saw, the, decade, herald, a, seri...","[The 1900s, the decade]"
6,"(1914, saw, the, completion, of, the, Panama, ...",[the Panama Canal]
7,"(The, Scramble, for, Africa, continued, in, th...","[Scramble, Africa, the 1900s]"
8,"(The, atrocities, in, the, Congo, Free, State,...",[the Congo Free State]
9,"(From, 1914, to, 1918, ,, the, First, World, W...","[1914 to 1918, the First World War]"


Filtering entities from article

In [17]:
# Function to filter out entities not of interest

def filter_entity(ent_list, alias_dict):
    return [ent for ent in ent_list if ent.lower() in alias_dict]

In [18]:
# Apply this to sentences df
df_sentences["country_entities"] = df_sentences["entities"].apply(
    lambda ents: filter_entity(ents, expanded_aliases)
)

In [19]:
filter_entity(["American","Zambia", "CF", "2"], expanded_aliases)

['American', 'Zambia']

In [20]:
df_sentences['country_entities'].head(20)

0                     [Spanish]
1                    [Japanese]
2                            []
3                            []
4                            []
5                            []
6                            []
7                            []
8                            []
9                            []
10                           []
11                           []
12    [France, Austria, Russia]
13            [Germany, Russia]
14                    [Germany]
15                           []
16          [Germany, American]
17                    [British]
18                           []
19                           []
Name: country_entities, dtype: object

In [21]:
# Filter out sentences with no country entities
df_sentences_filtered = df_sentences[df_sentences['country_entities'].map(len) > 0]

In [22]:
df_sentences_filtered.tail(10)

,sentence,entities,country_entities
279,"(In, the, 1990s, ,, work, on, the, Internation...","[the 1990s, the International Space Station, t...","[Russia, Japan]"
283,"(The, Sino, -, Soviet, split, had, removed, th...","[Sino-Soviet, USSR, the People's Republic of C...","[USSR, U.S.]"
284,"(Mikhail, Gorbachev, ,, its, last, leader, ,, ...","[Mikhail Gorbachev, Berlin, Soviet, Gorbachev,...",[Soviet]
285,"(Boris, Yeltsin, ,, president, of, Russia, ,, ...","[Boris Yeltsin, Russia]",[Russia]
320,"(The, people, of, the, Indian, subcontinent, ,...","[Indian, a sixth, the end of the century, firs...",[Indian]
321,"(China, ,, an, ancient, nation, comprising, a,...","[China, a fifth, West, East]",[China]
326,"(The, influence, of, China, and, India, was, a...","[China, India, West]","[China, India]"
331,"(Meanwhile, in, South, Africa, ,, the, aparthe...","[South Africa, Nelson Mandela, first, four and...",[South Africa]
332,"(In, Rwanda, ,, an, estimated, one, million, p...","[Rwanda, an estimated one million, Tutsis, Hut...",[Rwanda]
336,"(Despots, such, as, Kim, Jong, -, il, of, Nort...","[Kim Jong-il, North Korea]",[North Korea]


In [23]:
# map country entities to standardised name and lower case
def map_to_canonical(entities, alias_dict):
    return [alias_dict[ent.lower()] for ent in entities if ent.lower() in alias_dict]

In [24]:
df_sentences_filtered = df_sentences_filtered.copy()
df_sentences_filtered["country_canonicals"] = df_sentences_filtered["country_entities"].apply(
    lambda ents: map_to_canonical(ents, expanded_aliases)
)
df_sentences_filtered.tail(10)

,sentence,entities,country_entities,country_canonicals
279,"(In, the, 1990s, ,, work, on, the, Internation...","[the 1990s, the International Space Station, t...","[Russia, Japan]","[russia, japan]"
283,"(The, Sino, -, Soviet, split, had, removed, th...","[Sino-Soviet, USSR, the People's Republic of C...","[USSR, U.S.]","[russia, united states]"
284,"(Mikhail, Gorbachev, ,, its, last, leader, ,, ...","[Mikhail Gorbachev, Berlin, Soviet, Gorbachev,...",[Soviet],[russia]
285,"(Boris, Yeltsin, ,, president, of, Russia, ,, ...","[Boris Yeltsin, Russia]",[Russia],[russia]
320,"(The, people, of, the, Indian, subcontinent, ,...","[Indian, a sixth, the end of the century, firs...",[Indian],[india]
321,"(China, ,, an, ancient, nation, comprising, a,...","[China, a fifth, West, East]",[China],[china]
326,"(The, influence, of, China, and, India, was, a...","[China, India, West]","[China, India]","[china, india]"
331,"(Meanwhile, in, South, Africa, ,, the, aparthe...","[South Africa, Nelson Mandela, first, four and...",[South Africa],[south africa]
332,"(In, Rwanda, ,, an, estimated, one, million, p...","[Rwanda, an estimated one million, Tutsis, Hut...",[Rwanda],[rwanda]
336,"(Despots, such, as, Kim, Jong, -, il, of, Nort...","[Kim Jong-il, North Korea]",[North Korea],"[korea, democratic people's republic of]"


Create Relationships

In [25]:
# Defining relationships 

relationships = [] # create an empty list

for i in range(df_sentences_filtered.index[-1]):
    end_i = min(i+5, df_sentences_filtered.index[-1])
    country_list = sum((df_sentences_filtered.loc[i: end_i].country_canonicals), [])
    
    # Remove duplicated characters that are next to each other
    country_unique = [country_list[i] for i in range(len(country_list)) 
                   if (i==0) or country_list[i] != country_list[i-1]]
    
    if len(country_unique) > 1:
        for idx, a in enumerate(country_unique[:-1]):
            b = country_unique[idx + 1]
            relationships.append({"source": a, "target": b})

In [30]:
relationship_df = pd.DataFrame(relationships)

In [31]:
relationship_df

,source,target
0,spain,japan
1,france,austria
2,austria,russia
3,france,austria
4,austria,russia
...,...,...
1322,south africa,rwanda
1323,south africa,rwanda
1324,south africa,rwanda
1325,rwanda,"korea, democratic people's republic of"


In [32]:
# Sort the cases with a- >b and b- >a
relationships_df = pd.DataFrame(np.sort(relationship_df.values, axis = 1), 
                                columns = relationship_df.columns)
relationships_df.head(5)

,source,target
0,japan,spain
1,austria,france
2,austria,russia
3,austria,france
4,austria,russia


In [33]:
# Summarise the interactions and view sorted
relationships_df["value"] = 1
relationships_df = (
    relationships_df.groupby(["source","target"], 
                                          as_index=False)
    .sum()
    .sort_values("value", ascending=False)
)

relationships_df.head(10)

,source,target,value
27,china,japan,68
95,japan,united states,58
65,germany,united kingdom,54
61,germany,russia,53
59,germany,japan,48
54,france,united kingdom,45
116,russia,united states,40
90,japan,russia,37
66,germany,united states,32
48,france,germany,31


In [34]:
relationships_df.to_csv('20C_countries_relationships.csv')